# 03 - U-Net Model Implementation

## Objective
Build a U-Net architecture using PyTorch for microscopy image denoising.

## Architecture
1. Encoder
2. Bottleneck
3. Decoder
4. Skip Connections
5. Complete U-Net Model

## Input and Output
- Input: Noisy microscopy image
- Target: Clean ground-truth microscopy image
- Output: Denoised microscopy image

In [2]:
import torch
import torch.nn as nn

# Create a sample batch
x = torch.randn(8, 1, 256, 256)

print("Input shape:", x.shape)

# Define convolution layer
conv = nn.Conv2d(
    in_channels=1,
    out_channels=16,
    kernel_size=3,
    stride=1,
    padding=1
)

# Apply convolution
output = conv(x)

print("Output shape:", output.shape)

Input shape: torch.Size([8, 1, 256, 256])
Output shape: torch.Size([8, 16, 256, 256])


In [3]:
# Define ReLU activation
relu = nn.ReLU()

# Apply ReLU to convolution output
activated_output = relu(output)

print("Before ReLU:")
print("Minimum:", output.min().item())
print("Maximum:", output.max().item())

print("\nAfter ReLU:")
print("Minimum:", activated_output.min().item())
print("Maximum:", activated_output.max().item())

print("Output shape:", activated_output.shape)

Before ReLU:
Minimum: -3.3329989910125732
Maximum: 3.500631093978882

After ReLU:
Minimum: 0.0
Maximum: 3.500631093978882
Output shape: torch.Size([8, 16, 256, 256])


In [4]:
# Define Max Pooling
pool = nn.MaxPool2d(
    kernel_size=2,
    stride=2
)

# Apply pooling after ReLU
pooled_output = pool(activated_output)

print("Before pooling:", activated_output.shape)
print("After pooling:", pooled_output.shape)

Before pooling: torch.Size([8, 16, 256, 256])
After pooling: torch.Size([8, 16, 128, 128])


In [5]:
# Define upsampling
upsample = nn.Upsample(
    scale_factor=2,
    mode="bilinear",
    align_corners=False
)

# Apply upsampling
upsampled_output = upsample(pooled_output)

print("Before upsampling:", pooled_output.shape)
print("After upsampling:", upsampled_output.shape)

Before upsampling: torch.Size([8, 16, 128, 128])
After upsampling: torch.Size([8, 16, 256, 256])


In [6]:
# Simulate encoder and decoder feature maps

encoder_features = torch.randn(8, 16, 256, 256)

decoder_features = torch.randn(8, 16, 256, 256)

# Concatenate along channel dimension
combined_features = torch.cat(
    [encoder_features, decoder_features],
    dim=1
)

print("Encoder shape:", encoder_features.shape)
print("Decoder shape:", decoder_features.shape)
print("Combined shape:", combined_features.shape)

Encoder shape: torch.Size([8, 16, 256, 256])
Decoder shape: torch.Size([8, 16, 256, 256])
Combined shape: torch.Size([8, 32, 256, 256])


In [7]:
class DoubleConv(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

In [8]:
double_conv = DoubleConv(1, 64)

x = torch.randn(8, 1, 256, 256)

output = double_conv(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)

Input shape: torch.Size([8, 1, 256, 256])
Output shape: torch.Size([8, 64, 256, 256])


In [9]:
class EncoderBlock(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = DoubleConv(in_channels, out_channels)

        self.pool = nn.MaxPool2d(
            kernel_size=2,
            stride=2
        )

    def forward(self, x):

        features = self.conv(x)

        pooled = self.pool(features)

        return features, pooled

In [10]:
encoder = EncoderBlock(1, 64)

x = torch.randn(8, 1, 256, 256)

features, pooled = encoder(x)

print("Input:", x.shape)
print("Encoder features:", features.shape)
print("Pooled output:", pooled.shape)

Input: torch.Size([8, 1, 256, 256])
Encoder features: torch.Size([8, 64, 256, 256])
Pooled output: torch.Size([8, 64, 128, 128])


In [11]:
class Bottleneck(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x):
        return self.conv(x)

In [12]:
bottleneck = Bottleneck(64, 128)

x = torch.randn(8, 64, 128, 128)

output = bottleneck(x)

print("Input shape:", x.shape)
print("Bottleneck output:", output.shape)

Input shape: torch.Size([8, 64, 128, 128])
Bottleneck output: torch.Size([8, 128, 128, 128])


In [13]:
class DecoderBlock(nn.Module):

    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()

        self.up = nn.ConvTranspose2d(
            in_channels,
            out_channels,
            kernel_size=2,
            stride=2
        )

        self.conv = DoubleConv(
            out_channels + skip_channels,
            out_channels
        )

    def forward(self, x, skip):

        x = self.up(x)

        x = torch.cat([x, skip], dim=1)

        x = self.conv(x)

        return x

In [14]:
decoder = DecoderBlock(
    in_channels=128,
    skip_channels=64,
    out_channels=64
)

x = torch.randn(8, 128, 128, 128)

skip = torch.randn(8, 64, 256, 256)

output = decoder(x, skip)

print("Decoder input:", x.shape)
print("Skip connection:", skip.shape)
print("Decoder output:", output.shape)

Decoder input: torch.Size([8, 128, 128, 128])
Skip connection: torch.Size([8, 64, 256, 256])
Decoder output: torch.Size([8, 64, 256, 256])


In [15]:
class OutputLayer(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=1
        )

    def forward(self, x):
        return self.conv(x)

In [16]:
output_layer = OutputLayer(64, 1)

x = torch.randn(8, 64, 256, 256)

output = output_layer(x)

print("Input shape:", x.shape)
print("Final output shape:", output.shape)

Input shape: torch.Size([8, 64, 256, 256])
Final output shape: torch.Size([8, 1, 256, 256])


In [17]:
class UNet(nn.Module):

    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()

        # Encoder
        self.encoder1 = EncoderBlock(in_channels, 64)
        self.encoder2 = EncoderBlock(64, 128)
        self.encoder3 = EncoderBlock(128, 256)
        self.encoder4 = EncoderBlock(256, 512)

        # Bottleneck
        self.bottleneck = Bottleneck(512, 1024)

        # Decoder
        self.decoder4 = DecoderBlock(1024, 512, 512)
        self.decoder3 = DecoderBlock(512, 256, 256)
        self.decoder2 = DecoderBlock(256, 128, 128)
        self.decoder1 = DecoderBlock(128, 64, 64)

        # Final output
        self.output_layer = OutputLayer(64, out_channels)

    def forward(self, x):

        # Encoder
        skip1, x = self.encoder1(x)
        skip2, x = self.encoder2(x)
        skip3, x = self.encoder3(x)
        skip4, x = self.encoder4(x)

        # Bottleneck
        x = self.bottleneck(x)

        # Decoder with skip connections
        x = self.decoder4(x, skip4)
        x = self.decoder3(x, skip3)
        x = self.decoder2(x, skip2)
        x = self.decoder1(x, skip1)

        # Output
        return self.output_layer(x)

In [18]:
model = UNet(
    in_channels=1,
    out_channels=1
)

print(model)

UNet(
  (encoder1): EncoderBlock(
    (conv): DoubleConv(
      (conv): Sequential(
        (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): ReLU(inplace=True)
        (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (3): ReLU(inplace=True)
      )
    )
    (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (encoder2): EncoderBlock(
    (conv): DoubleConv(
      (conv): Sequential(
        (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): ReLU(inplace=True)
        (2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (3): ReLU(inplace=True)
      )
    )
    (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (encoder3): EncoderBlock(
    (conv): DoubleConv(
      (conv): Sequential(
        (0): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): ReLU(inpla

In [19]:
sample_input = torch.randn(2, 1, 256, 256)

with torch.no_grad():
    prediction = model(sample_input)

print("Input shape:", sample_input.shape)
print("Output shape:", prediction.shape)

Input shape: torch.Size([2, 1, 256, 256])
Output shape: torch.Size([2, 1, 256, 256])


In [20]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Total parameters: 31,030,593
Trainable parameters: 31,030,593


## 6. Model Preparation

In this section, we will:
1. Select the available computation device (CPU/GPU).
2. Move the U-Net model to that device.
3. Connect the training and validation DataLoaders.
4. Verify that the model can process an actual microscopy image batch.

In [21]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Selected device:", device)

model = model.to(device)

Selected device: cpu


## 7. Connecting the Dataset

In this section, we will connect the preprocessed microscopy dataset
to our U-Net model using PyTorch DataLoaders.

In [22]:
import os
import pandas as pd
import numpy as np
import tifffile
from pathlib import Path

from torch.utils.data import Dataset, DataLoader

In [23]:
# Locate project root
candidates = [Path.cwd(), Path.cwd().parent]

project_root = next(
    p for p in candidates
    if (p / "outputs/splits").exists()
)

# Load dataset splits
train_df = pd.read_csv(project_root / "outputs/splits/train.csv")
val_df = pd.read_csv(project_root / "outputs/splits/validation.csv")
test_df = pd.read_csv(project_root / "outputs/splits/test.csv")

print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Testing samples:", len(test_df))

Training samples: 1719
Validation samples: 369
Testing samples: 369


In [24]:
# Define image directories
base_path = project_root / "data" / "Archive" / "small_images"

noisy_path = base_path / "noisy"
gt_path = base_path / "gt"

# Reconstruct image paths
for df in [train_df, val_df, test_df]:
    df["noisy_path"] = df["filename"].apply(
        lambda name: str(noisy_path / name)
    )

    df["gt_path"] = df["filename"].apply(
        lambda name: str(gt_path / name)
    )

# Verify paths
print("First noisy image:")
print(train_df.iloc[0]["noisy_path"])

print("\nFirst ground-truth image:")
print(train_df.iloc[0]["gt_path"])

print("\nNoisy file exists:",
      os.path.exists(train_df.iloc[0]["noisy_path"]))

print("GT file exists:",
      os.path.exists(train_df.iloc[0]["gt_path"]))


First noisy image:
C:\Users\learn\Microscopy-Denoising\data\Archive\small_images\noisy\1881.tif

First ground-truth image:
C:\Users\learn\Microscopy-Denoising\data\Archive\small_images\gt\1881.tif

Noisy file exists: True
GT file exists: True


## 8. Creating the Custom PyTorch Dataset

Create a custom Dataset class to load paired noisy and ground-truth microscopy images.

In [25]:

class MicroscopyDataset(Dataset):

    def __init__(self, dataframe):
        self.dataframe = dataframe.reset_index(drop=True)

        self.noisy_scale = 78.0
        self.gt_scale = 2252.0

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        # Load noisy and ground-truth images
        noisy_image = tifffile.imread(row["noisy_path"])
        gt_image = tifffile.imread(row["gt_path"])

        # Normalize images
        noisy_image = noisy_image.astype(np.float32) / self.noisy_scale
        gt_image = gt_image.astype(np.float32) / self.gt_scale

        # Add channel dimension
        noisy_image = np.expand_dims(noisy_image, axis=0)
        gt_image = np.expand_dims(gt_image, axis=0)

        # Convert to PyTorch tensors
        noisy_tensor = torch.from_numpy(noisy_image)
        gt_tensor = torch.from_numpy(gt_image)

        return noisy_tensor, gt_tensor


In [26]:
# Create dataset object
train_dataset = MicroscopyDataset(train_df)

# Load first image pair
noisy_tensor, gt_tensor = train_dataset[0]

print("Noisy tensor shape:", noisy_tensor.shape)
print("GT tensor shape:", gt_tensor.shape)

print("Noisy tensor dtype:", noisy_tensor.dtype)
print("GT tensor dtype:", gt_tensor.dtype)

print("Noisy min/max:",
      noisy_tensor.min().item(),
      noisy_tensor.max().item())

print("GT min/max:",
      gt_tensor.min().item(),
      gt_tensor.max().item())

Noisy tensor shape: torch.Size([1, 256, 256])
GT tensor shape: torch.Size([1, 256, 256])
Noisy tensor dtype: torch.float32
GT tensor dtype: torch.float32
Noisy min/max: 0.0 1.3205127716064453
GT min/max: 0.05550621822476387 1.2078152894973755


## 9. Creating PyTorch DataLoaders

Create training, validation and testing DataLoaders to load microscopy image pairs in batches.

In [27]:

# Create dataset objects
train_dataset = MicroscopyDataset(train_df)
val_dataset = MicroscopyDataset(val_df)
test_dataset = MicroscopyDataset(test_df)

# Define batch size
BATCH_SIZE = 2

# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))


Training batches: 860
Validation batches: 185
Testing batches: 185


In [28]:
# Get one batch from training DataLoader
noisy_batch, gt_batch = next(iter(train_loader))

print("Noisy batch shape:", noisy_batch.shape)
print("GT batch shape:", gt_batch.shape)

print("Noisy batch dtype:", noisy_batch.dtype)
print("GT batch dtype:", gt_batch.dtype)

Noisy batch shape: torch.Size([2, 1, 256, 256])
GT batch shape: torch.Size([2, 1, 256, 256])
Noisy batch dtype: torch.float32
GT batch dtype: torch.float32


In [29]:
# Pass actual microscopy batch through U-Net

model.eval()

with torch.no_grad():
    noisy_batch = noisy_batch.to(device)
    prediction = model(noisy_batch)

print("Input shape:", noisy_batch.shape)
print("Prediction shape:", prediction.shape)
print("Prediction dtype:", prediction.dtype)

Input shape: torch.Size([2, 1, 256, 256])
Prediction shape: torch.Size([2, 1, 256, 256])
Prediction dtype: torch.float32


## 10. Loss Function and Optimizer

Define the loss function and optimizer for training the U-Net model.

- Loss Function: L1 Loss
- Optimizer: Adam

In [30]:
# Define loss function
criterion = nn.L1Loss()

# Define optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Loss function:", criterion)
print("Optimizer:", optimizer)

Loss function: L1Loss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


## 11. Training Function

Implement the training loop to:
1. Process batches of noisy microscopy images.
2. Generate denoised predictions.
3. Calculate L1 loss.
4. Perform backpropagation.
5. Update model parameters using Adam.

In [31]:

def train_one_epoch(model, loader, criterion, optimizer, device):

    model.train()

    running_loss = 0.0

    for noisy, gt in loader:

        noisy = noisy.to(device)
        gt = gt.to(device)

        # Reset gradients
        optimizer.zero_grad()

        # Forward pass
        prediction = model(noisy)

        # Calculate loss
        loss = criterion(prediction, gt)

        # Backpropagation
        loss.backward()

        # Update model parameters
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(loader)

    return epoch_loss


## 12. Validation Function

Evaluate the model on validation data without updating its parameters.

In [32]:

def validate_one_epoch(model, loader, criterion, device):

    model.eval()

    running_loss = 0.0

    with torch.no_grad():

        for noisy, gt in loader:

            noisy = noisy.to(device)
            gt = gt.to(device)

            # Forward pass
            prediction = model(noisy)

            # Calculate validation loss
            loss = criterion(prediction, gt)

            running_loss += loss.item()

    epoch_loss = running_loss / len(loader)

    return epoch_loss


## 13. Training Configuration

Define the number of epochs, checkpoint directory and training settings.

In [33]:

from pathlib import Path

# Training configuration
EPOCHS = 10
BATCH_SIZE = 2
LEARNING_RATE = 0.001

# Directory to save model checkpoints
checkpoint_dir = project_root / "outputs" / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

print("Epochs:", EPOCHS)
print("Batch size:", BATCH_SIZE)
print("Learning rate:", LEARNING_RATE)
print("Checkpoint directory:", checkpoint_dir)


Epochs: 10
Batch size: 2
Learning rate: 0.001
Checkpoint directory: C:\Users\learn\Microscopy-Denoising\outputs\checkpoints


## 14. Initial Training Test

Test one training iteration to verify forward propagation, loss calculation, backpropagation and optimizer updates.

In [34]:

# Take one batch
noisy, gt = next(iter(train_loader))

noisy = noisy.to(device)
gt = gt.to(device)

# Training mode
model.train()

# Reset gradients
optimizer.zero_grad()

# Forward pass
prediction = model(noisy)

# Calculate loss
loss = criterion(prediction, gt)

# Backpropagation
loss.backward()

# Update weights
optimizer.step()

print("Initial training loss:", loss.item())
print("One training iteration completed successfully!")


Initial training loss: 0.08105935156345367
One training iteration completed successfully!


In [35]:
torch.save(
    model.state_dict(),
    checkpoint_dir / "unet_initial_test.pth"
)

print("Initial checkpoint saved successfully!")

Initial checkpoint saved successfully!


In [37]:
print("Model exists:", "model" in globals())
print("Optimizer exists:", "optimizer" in globals())
print("Loss function exists:", "criterion" in globals())
print("Training loader exists:", "train_loader" in globals())
print("Validation loader exists:", "val_loader" in globals())
print("Device:", device if "device" in globals() else "Not defined")
print("Epochs:", EPOCHS if "EPOCHS" in globals() else "Not defined")

Model exists: True
Optimizer exists: True
Loss function exists: True
Training loader exists: True
Validation loader exists: True
Device: cpu
Epochs: 10


In [38]:
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

print("Model parameters:",
      sum(p.numel() for p in model.parameters()))

print("Checkpoint exists:",
      (checkpoint_dir / "unet_initial_test.pth").exists())

Training batches: 860
Validation batches: 185
Model parameters: 31030593
Checkpoint exists: True


In [39]:
import time

model.train()

start_time = time.time()

for batch_idx, (noisy, gt) in enumerate(train_loader):
    if batch_idx == 5:
        break

    noisy = noisy.to(device)
    gt = gt.to(device)

    optimizer.zero_grad()

    prediction = model(noisy)
    loss = criterion(prediction, gt)

    loss.backward()
    optimizer.step()

elapsed = time.time() - start_time

print("5 batches completed!")
print("Time taken:", round(elapsed, 2), "seconds")
print("Average time per batch:", round(elapsed / 5, 2), "seconds")
print("Estimated training time per epoch:",
      round((elapsed / 5) * len(train_loader) / 60, 2), "minutes")

5 batches completed!
Time taken: 32.77 seconds
Average time per batch: 6.55 seconds
Estimated training time per epoch: 93.94 minutes
